# File Prober

This notebook probes unprocessed video files in the database, queries media information using ffprobe, and displays results.

Workflow:
1. Query all files that exist in the database but have no encode record yet
2. Run ffprobe on each file to extract metadata (resolution, codec, duration, etc.)
3. Display results and handle errors gracefully

In [1]:
import sqlite3
import subprocess
import json
from pathlib import Path

print("Libraries imported successfully")

Libraries imported successfully


## Define File Probing Functions

In [2]:
def run_ffprobe(file_path):
    try:
        cmd = [
            'ffprobe',
            '-loglevel', 'quiet',
            '-show_entries', 'format:stream=index,stream,codec_type,codec_name,channel_layout,format=nb_streams',  
            '-of', 'json',
            file_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        
        if result.returncode != 0:
            return {'error': f'ffprobe failed: {result.stderr}'}
        
        probe_data = json.loads(result.stdout)
        
        # Pretty-print the ffprobe JSON output
        print(json.dumps(probe_data, indent=2))
        
        return probe_data
    
    except FileNotFoundError:
        return {'error': 'ffprobe not found. Ensure ffmpeg is installed and in PATH.'}
    except subprocess.TimeoutExpired:
        return {'error': 'ffprobe timeout (file too large or network issue)'}
    except json.JSONDecodeError:
        return {'error': 'Invalid ffprobe JSON output'}
    except Exception as e:
        return {'error': str(e)}


# Check Codecs
Loops through the streams in stream_info from requires_encoding, then calls functions to determine if the steam needs encoding based on stream type conditions 

Todo:
- [x] Copy over the stream looping function from Boilest v1.0
- [ ] Research SVT-AV1 best practices for various media types
- [ ] Store SVT-AV1 best practice presets in the DB
- [ ] Call best-practive presets in check_video_stream
- [ ] Determine what audio codec to go with
- [ ] Determine what the compromises will be if ASS subtitles are re-encoded as SubRip
- [ ] Determine if there are consequences for deleting attachments 

In [ ]:
def check_codecs(encoding_decision,stream_info, ffmpeg_command):
    streams_count = stream_info['format']['nb_streams']
    
    
    #print('There are : ' + str(streams_count) + ' streams')
    
    for i in range (0,streams_count):
        codec_type = stream_info['streams'][i]['codec_type'] 
        if codec_type == 'video':
            print('Stream ' + str(i) + ' is video')
            encoding_decision, ffmpeg_command = check_video_stream(encoding_decision, i, stream_info, ffmpeg_command)
        elif codec_type == 'audio':
            encoding_decision, ffmpeg_command = check_audio_stream(encoding_decision, i, stream_info, ffmpeg_command)
            print('audio stream')
        elif codec_type == 'subtitle':
            encoding_decision, ffmpeg_command = check_subtitle_stream(encoding_decision, i, stream_info, ffmpeg_command)
            print('subtitle stream')
        elif codec_type == 'attachment':
            encoding_decision, ffmpeg_command = check_attachmeent_stream(encoding_decision, i, stream_info, ffmpeg_command) 
            print('attachment stream')    
    print (encoding_decision)   
    print (ffmpeg_command)
    return encoding_decision, ffmpeg_command

def check_video_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the video stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    desired_video_codec = 'av1'
    print('Steam ' + str(i) + ' codec is: ' + codec_name)
    if codec_name == desired_video_codec:
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v copy'
    elif codec_name == 'mjpeg':
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v copy'
    elif codec_name != desired_video_codec: 
        encoding_decision = True
        svt_av1_string = "libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise=0:tune=0:enable-qm=1:qm-min=0:qm-max=15"
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v ' + svt_av1_string
    else:
        print('ignoring for now')
    return encoding_decision, ffmpeg_command


def check_audio_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the audio stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    # This will be populated at a later date
    #desired_audio_codec = 'aac'
    #if codec_name != desired_video_codec:
    #    encoding_decision = True
    print('Steam ' + str(i) + ' codec is: ' + codec_name)
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:a copy'
    return encoding_decision, ffmpeg_command


def check_subtitle_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the subtitle stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    # This will be populated at a later date
    #desired_subtitle_codec = 'srt'
    #if codec_name != desired_subtitle_codec:
    #    encoding_decision = True
    print('Steam ' + str(i) + ' codec is: ' + codec_name)
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:s copy'
    return encoding_decision, ffmpeg_command


def check_attachmeent_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the attachment stream from check_codecs to determine if the stream needs encoding
    # This will be populated at a later date
    #desired_attachment_codec = '???'
    #if codec_name != desired_attachment_codec:
    #    encoding_decision = True
    # Note, attachments may not have a codec name if the attachment is an image
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:t copy'
    return encoding_decision, ffmpeg_command

In [4]:
import os

def output_file_name(file_path, needs_encoding):
    # Get the filename and current extension
    filename = os.path.basename(file_path)
    name_without_ext = os.path.splitext(filename)[0]
    current_ext = os.path.splitext(filename)[1]
    
    # Change extension to .mkv if it's not already
    if current_ext.lower() != '.mkv':
        new_filename = name_without_ext + '.mkv'
        needs_encoding = True
    else:
        new_filename = filename
    
    # Return just the new filename without any directory path
    return new_filename, needs_encoding

In [5]:
def get_file_size_kb(file_path):
    try:
        file_size_bytes = Path(file_path).stat().st_size
        file_size_kb = int(file_size_bytes / 1024)
        return file_size_kb
    except FileNotFoundError:
        print(f"✗ File not found: {file_path}")
        return 0
    except Exception as e:
        print(f"✗ Error getting file size: {e}")
        return 0

In [6]:
import sqlite3
from datetime import datetime


def write_to_encode(file_path, guid):
    try:
        # Use provided guid
        directory_path = os.path.dirname(file_path)
        input_file_name = os.path.basename(file_path)

        # Get file size in KB
        before_file_size = get_file_size_kb(file_path)

        # Probe file and determine encoding via check_codecs
        probe_data = run_ffprobe(file_path)
        ffmpeg_command = ""
        needs_encoding = False

        if isinstance(probe_data, dict) and 'error' in probe_data:
            print(f"Error probing file: {probe_data['error']}")
            # Keep defaults: needs_encoding=False, ffmpeg_command=""
        else:
            needs_encoding, ffmpeg_command = check_codecs(needs_encoding, probe_data, ffmpeg_command)

        # Only populate output_file and ffmpeg_string if needs_encoding is True
        if needs_encoding:
            output_file, _ = output_file_name(file_path, needs_encoding)
            ffmpeg_string = ffmpeg_command
        else:
            output_file = None
            ffmpeg_string = None

        # Get current datetime
        date_added = datetime.now().isoformat()

        # Convert decision to string
        decision = str(needs_encoding)

        # Write to database
        db_path = 'boilest.db'
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()

        cur.execute("""
            INSERT INTO encode (guid, directory_path, input_file_name, output_file_name, 
                               before_file_size, decision, ffmpeg_string, date_added)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (guid, directory_path, input_file_name, output_file, before_file_size, decision, ffmpeg_string, date_added))

        conn.commit()
        conn.close()

        print(f"✓ Successfully wrote encode record for {input_file_name}")

        return {
            'guid': guid,
            'directory_path': directory_path,
            'input_file_name': input_file_name,
            'output_file_name': output_file,
            'before_file_size': before_file_size,
            'decision': decision,
            'ffmpeg_string': ffmpeg_string,
            'date_added': date_added
        }

    except Exception as e:
        print(f"✗ Error writing to encode table: {e}")
        return None


In [ ]:
def enqueue_all_files_for_encoding(db_path='boilest.db'):
    results = []
    skip_guids = set()
    processed = 0

    while True:
        conn = None
        row = None
        try:
            conn = sqlite3.connect(db_path)
            cur = conn.cursor()

            if skip_guids:
                placeholders = ",".join(["?"] * len(skip_guids))
                cur.execute(
                    f"""
                    SELECT f.guid, f.file_path
                    FROM files f
                    LEFT JOIN encode e ON e.guid = f.guid
                    WHERE e.guid IS NULL AND f.guid NOT IN ({placeholders})
                    LIMIT 1
                    """,
                    tuple(skip_guids),
                )
            else:
                cur.execute(
                    """
                    SELECT f.guid, f.file_path
                    FROM files f
                    LEFT JOIN encode e ON e.guid = f.guid
                    WHERE e.guid IS NULL
                    LIMIT 1
                    """
                )

            row = cur.fetchone()
        except Exception as e:
            print(f"✗ Error querying next file: {e}")
            break
        finally:
            if conn:
                conn.close()

        if not row:
            print(f"✓ No more files requiring encode. Processed {processed} file(s).")
            break

        guid, file_path = row

        if not file_path:
            print(f"✗ Missing file path for guid {guid}")
            skip_guids.add(guid)
            results.append({"guid": guid, "file_path": file_path, "error": "missing file_path"})
            continue

        if not Path(file_path).exists():
            print(f"✗ File not found on disk for guid {guid}: {file_path}")
            skip_guids.add(guid)
            results.append({"guid": guid, "file_path": file_path, "error": "file not found on disk"})
            continue

        result = write_to_encode(file_path, guid)
        results.append({"guid": guid, "file_path": file_path, "write_result": result})
        processed += 1

    print(f"✓ Attempted encode writes for {len(results)} file(s)")
    return results


In [10]:
enqueue_all_files_for_encoding(db_path='boilest.db')


✓ No more files requiring encode. Processed 0 file(s).
✓ Attempted encode writes for 0 file(s)


[]